# Problem Statement

Create a Delta Live Tables pipeline that ingests Kafka messages into a bronze table, transforms and validates them into a silver table, and produces aggregated outputs in a gold table.

# Requirements

- Use Databricks Delta Live Tables with PySpark.
- Ingest from Kafka into a bronze layer.
- Parse and clean records in a silver layer.
- Aggregate curated data in a gold layer.
- Include clear schema handling and basic data quality filtering.
- Keep the notebook organized for DLT deployment.

# Imports

The next code cell imports the required PySpark and Delta Live Tables modules.

# Implementation

The next code cells define reusable parsing logic and DLT tables for bronze, silver, and gold layers.

# Example Usage

Deploy this notebook as a Delta Live Tables pipeline in Databricks, configure Kafka connection options in the pipeline settings, and run the pipeline to materialize the bronze, silver, and gold tables.

# Results

After execution, the pipeline will produce:

- A bronze table with raw Kafka payloads.
- A silver table with parsed and validated event records.
- A gold table with aggregated metrics derived from the silver layer.

In [ ]:
from typing import Dict

import dlt
from pyspark.sql import DataFrame, Column
from pyspark.sql import functions as F
from pyspark.sql import types as T

# Define the expected business schema for Kafka payloads.
EVENT_SCHEMA = T.StructType(
    [
        T.StructField('event_id', T.StringType(), True),
        T.StructField('event_type', T.StringType(), True),
        T.StructField('user_id', T.StringType(), True),
        T.StructField('event_ts', T.TimestampType(), True),
        T.StructField('value', T.DoubleType(), True),
    ]
)


def parse_event_payload(payload_col: Column) -> DataFrame:
    """Parse a JSON payload column into structured event fields."""
    parsed = F.from_json(payload_col.cast('string'), EVENT_SCHEMA)
    return parsed


@dlt.table(
    name='bronze_kafka_events',
    comment='Raw Kafka messages ingested into the bronze layer.'
)
def bronze_kafka_events() -> DataFrame:
    return (
        spark.readStream.format('kafka')
        .option('kafka.bootstrap.servers', 'KAFKA_BOOTSTRAP_SERVERS')
        .option('subscribe', 'KAFKA_TOPIC')
        .option('startingOffsets', 'latest')
        .load()
        .select(
            F.col('timestamp').alias('kafka_timestamp'),
            F.col('topic'),
            F.col('partition'),
            F.col('offset'),
            F.col('key').cast('string').alias('key'),
            F.col('value').cast('string').alias('value'),
        )
    )


@dlt.table(
    name='silver_events',
    comment='Cleaned and validated Kafka events in the silver layer.'
)
@dlt.expect_or_drop('valid_event_id', 'event_id IS NOT NULL')
@dlt.expect_or_drop('valid_event_type', 'event_type IS NOT NULL')
@dlt.expect_or_drop('valid_user_id', 'user_id IS NOT NULL')
@dlt.expect_or_drop('valid_event_ts', 'event_ts IS NOT NULL')
def silver_events() -> DataFrame:
    bronze_df = dlt.read_stream('bronze_kafka_events')
    parsed = parse_event_payload(F.col('value'))

    return bronze_df.select(
        'kafka_timestamp',
        'topic',
        'partition',
        'offset',
        'key',
        parsed.alias('event')
    ).select(
        'kafka_timestamp',
        'topic',
        'partition',
        'offset',
        'key',
        F.col('event.event_id').alias('event_id'),
        F.col('event.event_type').alias('event_type'),
        F.col('event.user_id').alias('user_id'),
        F.col('event.event_ts').alias('event_ts'),
        F.col('event.value').alias('value'),
    ).where(F.col('event_id').isNotNull())


@dlt.table(
    name='gold_event_metrics',
    comment='Aggregated event metrics in the gold layer.'
)
def gold_event_metrics() -> DataFrame:
    silver_df = dlt.read_stream('silver_events')

    return (
        silver_df.withWatermark('event_ts', '1 day')
        .groupBy(
            F.window('event_ts', '1 day'),
            F.col('event_type')
        )
        .agg(
            F.count('*').alias('event_count'),
            F.countDistinct('user_id').alias('distinct_users'),
            F.sum('value').alias('total_value'),
        )
        .select(
            F.col('window.start').alias('window_start'),
            F.col('window.end').alias('window_end'),
            'event_type',
            'event_count',
            'distinct_users',
            'total_value',
        )
    )